# TD 2 : récupérer et préparer une base

Cours de M1 Économie de l'innovation, Université de Picardie Jules Verne

**La question du TD.** Comment passer d'une source en ligne à une table propre, prête pour l'analyse.

| | |
|---|---|
| En séance (1 h, accompagné) | parties 1 à 3 |
| À distance (environ 3 h 30) | parties 4 à 7 |
| Jalon du projet | la note de cadrage d'une page |
| Ensuite | TD 3 : statistiques descriptives et comparaisons |

### Comment travailler avec ce notebook

Chaque partie porte un niveau. Avancez à votre rythme.

- **Essentiel** : à faire par tout le monde. Ces parties suffisent pour le projet.
- **Pour aller plus loin** : pour celles et ceux qui sont à l'aise avec Python.
- **Bonus** : des outils plus avancés (régression, probit, téléchargement direct). Les utiliser dans le projet est un plus. Ne pas les utiliser n'est jamais pénalisé.

Une cellule de code s'exécute en cliquant dessus puis en appuyant sur **Maj + Entrée**. Les zones **À vous** sont à compléter : une ligne de code ou quelques phrases. Une erreur n'est pas grave : lisez la dernière ligne du message, corrigez et relancez la cellule.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Trouver une base (Essentiel)

Les principales sources ouvertes sont décrites dans l'annexe sur les bases de données :

- Eurostat : R&D, enquête communautaire sur l'innovation (CIS), usage de l'IA par les entreprises,
- Banque mondiale et OCDE : séries par pays sur la R&D, les brevets, la diffusion des technologies,
- OMPI et PatentsView : brevets,
- EU Industrial R&D Investment Scoreboard : les entreprises qui investissent le plus en R&D.

Avant de télécharger, vérifiez quatre points dans la documentation de la source : la définition de l'indicateur, son unité (euros, % du PIB, nombre), l'unité d'observation (pays-année, entreprise, brevet) et la période couverte.

**Télécharger à la main.** Sur le site de la Banque mondiale (DataBank) ou d'Eurostat (Data Browser), choisissez l'indicateur, les pays et les années puis exportez au format CSV. Placez le fichier dans le dossier `donnees`.

**À vous.** Pour l'une de vos trois problématiques, notez l'indicateur, la source, l'unité et la période que vous comptez utiliser.

*Votre réponse :*

## 2. Lire un fichier (Essentiel)

Le fichier `brevets-pays-2000-2023.csv` contient pour les 19 pays du TD 1 le nombre de demandes de brevets déposées par des résident.es, la population et la part des produits de haute technologie dans les exportations de produits manufacturés.

In [ ]:
brevets = pd.read_csv("donnees/brevets-pays-2000-2023.csv")
print(brevets.shape)
brevets.head()

In [ ]:
# Le type de chaque colonne : nombre entier, nombre décimal ou texte
brevets.dtypes

Les fichiers diffèrent selon les sources. Trois réglages règlent la plupart des problèmes de lecture :

- un fichier français sépare souvent les colonnes par un point-virgule et les décimales par une virgule : `pd.read_csv(fichier, sep=";", decimal=",")`,
- un fichier qui commence par quelques lignes de commentaires : `pd.read_csv(fichier, skiprows=4)`,
- un fichier Excel : `pd.read_excel(fichier)`.

Si une colonne de nombres apparaît comme du texte (`object`), c'est souvent l'un de ces réglages qui manque.

## 3. Préparer la table (Essentiel)

Préparer une base, c'est sélectionner ce qui sert, nommer clairement et contrôler. On garde ici les années 2010 à 2023.

In [ ]:
# Garder les années récentes
prep = brevets[brevets["annee"] >= 2010].copy()

# Valeurs manquantes par colonne
prep.isna().sum()

Le nombre de brevets dépend de la taille du pays. Pour comparer, on le rapporte à la population. Créer une variable revient à écrire une opération entre colonnes.

In [ ]:
prep["brevets_par_million"] = prep["brevets_residents"] / prep["population"] * 1_000_000
prep[["code", "annee", "brevets_residents", "brevets_par_million"]].head()

**À vous.** Classez les pays selon le nombre de brevets par million d'habitant.es en 2021. Comparez avec le classement en nombre de brevets. Expliquez pourquoi les deux classements diffèrent et lequel convient pour comparer des pays.

*Votre réponse :*

In [ ]:
# À compléter : sélectionner l'année 2021 puis trier

## 4. Réunir deux tables (Pour aller plus loin)

Pour relier la R&D aux brevets, on réunit la base du TD 1 et la table des brevets. La **clé** est ce qui identifie une ligne dans les deux tables : ici le code du pays et l'année.

In [ ]:
rd = pd.read_csv("donnees/rd-pays-2000-2023.csv")
reunie = rd.merge(brevets, on=["code", "annee"], how="left")

# Contrôle : le nombre de lignes ne doit pas changer
print(rd.shape, reunie.shape)
reunie.head()

**À vous.** Expliquez pourquoi on contrôle le nombre de lignes après la fusion. Dites ce qui se passerait si une année apparaissait deux fois pour un même pays dans la table des brevets.

*Votre réponse :*

## 5. Un graphique soigné (Essentiel)

Un graphique de projet a un titre, des axes avec leurs unités, une légende et une source.

In [ ]:
choix = ["FRA", "DEU", "KOR", "USA"]
for code in choix:
    d = prep[prep["code"] == code]
    plt.plot(d["annee"], d["brevets_par_million"], label=code)

plt.title("Demandes de brevets déposées par des résident.es")
plt.xlabel("Année")
plt.ylabel("Brevets par million d'habitant.es")
plt.legend()
plt.figtext(0.01, -0.03, "Source : Banque mondiale, WDI (IP.PAT.RESD, SP.POP.TOTL)", fontsize=8)
plt.show()

**À vous.** Commentez le graphique en trois phrases. Rappelez ce qu'un brevet mesure et ce qu'il ne mesure pas (cf. Chapitre 3).

*Votre réponse :*

## 6. Télécharger directement depuis Eurostat (Bonus)

Plutôt que d'exporter un fichier à la main, on peut lire une table Eurostat directement par son adresse. La table `rd_e_gerdtot` donne les dépenses de R&D. L'adresse précise l'unité (`PC_GDP`, en % du PIB) et les pays. Il faut une connexion Internet.

In [ ]:
url = ("https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
       "rd_e_gerdtot/A.TOTAL.PC_GDP.FR+DE+IT+ES?format=SDMX-CSV&startPeriod=2015")
try:
    eurostat = pd.read_csv(url)
    eurostat = eurostat[["geo", "TIME_PERIOD", "OBS_VALUE"]]
    print(eurostat.head())
except Exception as erreur:
    print("Téléchargement impossible :", erreur)

**À vous.** Comparez la valeur d'Eurostat pour la France en 2021 avec celle de la Banque mondiale (TD 1). Si elles diffèrent, cherchez une explication dans la documentation des deux sources.

*Votre réponse :*

## 7. Pour le projet (Essentiel)

Rédigez votre note de cadrage d'une page :

1. votre question, rédigée selon le gabarit du Chapitre 13,
2. le mécanisme du cours sur lequel elle repose,
3. la ou les sources, avec le lien et la date de consultation,
4. les indicateurs envisagés avec leur unité et un graphique que vous comptez produire,
5. la principale limite que vous anticipez.

Chargez ensuite votre propre base dans un nouveau notebook en reprenant les parties 2 et 3.

---
Données : Banque mondiale, World Development Indicators, téléchargées en novembre 2026. Fichiers dans le dossier `donnees` (description dans `donnees/LISEZMOI.md`).